# CountConformingBitmasks

[Codility task](https://app.codility.com/programmers/trainings/9/count_conforming_bitmasks/) - Exercise 9, Bitwise operations - *medium*

We consider unsigned 30-bit integers (`0 <= B < 2**30`). Integer `A` **conforms** to `B` if `A` has a 1 in every position where `B` has a 1.

`solution(A, B, C)` must return how many unsigned 30-bit integers conform to **at least one** of `A`, `B`, `C`.

Example: `A = 1073741727`, `B = 1073741631`, `C = 1073741679` gives `8`.

Assumption: `A`, `B`, `C` are in `[0..1,073,741,823]`.

## Idea: inclusion-exclusion

- Mask `X` forces `popcount(X)` bits to 1; the other `30 - popcount(X)` bits are free. So the number of conformers is
  `f(X) = 2 ** (30 - popcount(X))`.
- Being a conformer of both `X` and `Y` means having all bits of `X | Y` set, i.e. conforming to `X | Y`.
- Union of three sets:

  `|A u B u C| = f(A) + f(B) + f(C) - f(A|B) - f(A|C) - f(B|C) + f(A|B|C)`

Time and space O(1). Iterating over all `2**30` candidates would take minutes.

In [ ]:
BITS = 30


def popcount(x):
    return bin(x).count("1")      # int.bit_count() exists on Python >= 3.10


def conformers(mask, bits=BITS):
    """How many `bits`-bit integers conform to `mask`."""
    return 2 ** (bits - popcount(mask))


def solution(A, B, C, bits=BITS):
    f = lambda x: conformers(x, bits)
    return (
        f(A) + f(B) + f(C)
        - f(A | B) - f(A | C) - f(B | C)
        + f(A | B | C)
    )

## Official example

In [ ]:
A, B, C = 1_073_741_727, 1_073_741_631, 1_073_741_679
assert solution(A, B, C) == 8

# the eight conformers listed in the statement
listed = {
    1_073_741_631, 1_073_741_679, 1_073_741_695, 1_073_741_727,
    1_073_741_759, 1_073_741_791, 1_073_741_807, 1_073_741_823,
}
conforms = lambda a, b: a & b == b
# All conformers have the top 22 bits set, so scanning the last 2**10 values finds them all.
found = {n for n in range(2**30 - 2**10, 2**30) if conforms(n, A) or conforms(n, B) or conforms(n, C)}
assert found == listed
print("official example OK ->", solution(A, B, C))

## Edge cases

In [ ]:
FULL = 2**30 - 1

assert solution(0, 0, 0) == 2**30                  # every integer conforms to 0
assert solution(FULL, FULL, FULL) == 1             # only 2**30 - 1 itself
assert solution(0, FULL, 5) == 2**30               # 0 already covers everything
assert solution(1, 1, 1) == 2**29                  # same mask three times
assert solution(1, 2, 4) == 2**30 - 2**27          # see the note below
print("edge cases OK")

Note on the last assertion: an integer conforms to `1`, `2` or `4` unless **none** of bits 0, 1, 2 is set. That leaves 27 free bits, so `2**27` integers fail all three, and the union has `2**30 - 2**27`.

## Brute-force check

With `bits` as a parameter we can verify the formula by enumeration on a small word size (all triples for `bits = 4`, random triples for `bits = 10`).

In [ ]:
import itertools
import random


def brute(A, B, C, bits):
    conforms = lambda a, b: a & b == b
    return sum(
        1 for n in range(2**bits)
        if conforms(n, A) or conforms(n, B) or conforms(n, C)
    )


# exhaustive for 4-bit words: 16**3 = 4096 triples
for A, B, C in itertools.product(range(2**4), repeat=3):
    assert solution(A, B, C, bits=4) == brute(A, B, C, 4), (A, B, C)

# random for 10-bit words
random.seed(0)
for _ in range(300):
    A, B, C = (random.randrange(2**10) for _ in range(3))
    assert solution(A, B, C, bits=10) == brute(A, B, C, 10), (A, B, C)
print("formula matches brute force")

## Notes

- Inclusion-exclusion works because `conform(X) and conform(Y)  <=>  conform(X | Y)`.
- The `bits` parameter is only for testing; Codility always uses 30.
- Result range: at most `2**30`, no overflow issues in Python.
- O(1) time and space.